In [9]:
OPEN_GOALS = [
    "You want to inform yourself what to do in case of an emergency during travel.",
    "You want more information about how to plan a research semester.",
    "You want to know how to book a flight",
    "You want to know how to book a hotel"
    ]

In [17]:
transcript = {}
answer_quality = {}
dialog_length = {}
per_user_dialog_length = {}
per_user_answer_quality = {}
dialog_success = {}
actual_dialog_length = {}
user_conditions = {}

current_user_goal = {}

# Read in user log
with open("generated_4/user_log.txt", "r") as user_info:
    for line in user_info:
        if "GROUP" in line:
            user, policy = line.split("||")
            user = user.split(":")[1].strip()
            policy = policy.split(":")[1].strip()
            user_conditions[user] = policy

# Read in chat log
with open("generated_4/chat_log.txt", "r") as chat_log:
    for line in chat_log:
        if "NEW DIALOG STARTED" in line:
            user = line.split("USER")
            user = user[1].strip()[:-4].strip()
            # start a new entry every time the system sends the new Dialog started signal
            if user in transcript:
                transcript[user].append([f"USER: {user} (POLICY: {user_conditions[user]})"])
            else:
                transcript[user] = [[f"USER: {user} (POLICY: {user_conditions[user]})"]]
        elif "GOAL" in line and not "GOAL ONCE" in line and not "REACHED GOAL" in line:
            # record the text shown to the user and the text from the corresponding goal node
            if "USER" in line:
                user, goal, _ = line.split("||")
                user = user.split(":")[1].strip()
                goal = goal.split(":")[1].strip()
                transcript[user][-1].append(f"GOAL TEXT: {goal}")
                current_user_goal[user] = goal
            else:
                user, goal = line.split("GOAL:")
                user = user.split("-")[0].strip()
                goal = goal.strip()
                transcript[user][-1].append(f"GOAL NODE: {goal}")
        elif "RESET" in line:
            # If the user clicks on restart, record that they've restarted
            user = line.split()[0].split("-")[0].strip()
            if len(transcript[user][-1]) > 0:
                transcript[user][-1].append("RESET")
        elif "INITIAL UTTERANCE" in line:
            # Initial utterances were logged differently, but are added to transcript like any other utterance
            user, utterance = line.split("INITIAL UTTERANCE:")
            user = user.split("-")[0].strip()
            utterance = utterance.strip()
            transcript[user][-1].append(f"USER: {utterance}")
        elif "USER UTTERANCE" in line:
            # TODO: fix how we record variable nodes
            if "VARIABLE NODE" in line:
                user = line.split("-")[0].strip()
                utterance = "(PRE-NLU) " + line.split("NLU:")[1].strip()
            else:
                user, utterance = line.split("->")
                user, dialog_num = user.split("-")
                user = user.strip()
                dialog_num = dialog_num.strip("$").strip()
                utterance = f"{utterance[16:].strip()}"
                if "(PRE-NLU)" in transcript[user][-1][-1]:
                    utterance = "(POST-NLU) " + utterance
            if len(utterance) > 0:
                transcript[user][-1].append(f"USER: {utterance}")
        elif "SKIP + ASK" in line:
            user = line.split("-")[0].strip()
            tmp = line.split("Node")[1].split(",")
            node = tmp[0].strip()
            utterance = "-".join(tmp[1:]).strip()
            transcript[user][-1].append(f"SYSTEM: (NODE: {node}) {utterance}")
        elif "ASKING" in line:
            tmp = line.split("-")
            user = tmp[0].strip()
            node = tmp[2].strip()
            utterance = "-".join(tmp[3:]).strip()
            if "(PRE-NLU)" in transcript[user][-1][-1]:
                before = transcript[user][-1].pop()
                transcript[user][-1].append(f"SYSTEM: (NODE: {node}) {utterance}")
                transcript[user][-1].append(before)
            else:
                transcript[user][-1].append(f"SYSTEM: (NODE: {node}) {utterance}")
        elif "USER" in line and "LENGTH" in line:
            user, rating = line.split("||")
            user = user.split(":")[1].strip()
            rating = int(rating.split(":")[1].strip())
            transcript[user][-1].append(f"SUBJECTIVE LENGTH: {rating}")
            policy = user_conditions[user]
            if policy not in dialog_length:
                dialog_length[policy] = []
            if user not in per_user_dialog_length:
                per_user_dialog_length[user] = []
            per_user_dialog_length[user].append(rating)
            dialog_length[policy].append(rating)
        elif "USER" in line and "QUALITY" in line:
            user, rating = line.split("||")
            user = user.split(":")[1].strip()
            rating = int(rating.split(":")[1].strip())
            transcript[user][-1].append(f"SUBJECTIVE QUALITY: {rating}")
            policy = user_conditions[user]
            if policy not in answer_quality:
                answer_quality[policy] = []
            answer_quality[policy].append(rating)
            if user not in per_user_answer_quality:
                per_user_answer_quality[user] = []
            per_user_answer_quality[user].append(rating)
        elif "ASKED GOAL ONCE" in line:
            user, reached = line.split("=>")
            user = user.split("-")[0].strip()
            policy = user_conditions[user]
            reached = reached.split(":")[1].strip()
            if reached == "True" and "DIALOG END: SUCCESS" not in transcript[user][-1][-3:]:
                transcript[user][-1].append("DIALOG END: SUCCESS")
                if policy not in dialog_success:
                    dialog_success[policy] = []
                dialog_success[policy].append(1)
            elif reached == "False" and "DIALOG END: FAILURE" not in transcript[user][-1][-3:] and current_user_goal[user]not  in OPEN_GOALS:
                transcript[user][-1].append("DIALOG END: FAILURE")
                if policy not in dialog_success:
                    dialog_success[policy] = []
                dialog_success[policy].append(0)
        elif "PERCIEVED LENGTH" in line:
            user, length = line.split("=>")
            user = user.split("-")[0].strip()
            policy = user_conditions[user]
            length = length.split(":")[1].strip()
            if "DIALOG LENGTH" not in transcript[user][-1][-1]:
                transcript[user][-1].append(f"DIALOG LENGTH: {length}")
                if policy not in actual_dialog_length:
                    actual_dialog_length[policy] = []
                actual_dialog_length[policy].append(int(length))

### Format cleaned transcript and save it to a new file

In [18]:
with open("generated_4/transcript.txt", "w") as outfile:
    for user in transcript:
        if len(transcript[user]) != 3:
            print(f"{user}: {len(transcript[user])}")
            continue
        for dialog in transcript[user]:
            # Only log if the user ever said anything
            if len(dialog) > 3:
                for line in dialog:
                    outfile.write(f"{line}\n")
                outfile.write("\n")

dbf6da8ef1042c4ccc7a717062c6a7: 6
